# 🚬 흡연 분류 V8 - Stacking 앙상블 (최종)

## 핵심 전략
- ✅ **앙상블 다양성**: XGBoost + LightGBM + CatBoost + **ExtraTrees** (배깅 추가)
- ✅ **Stacking**: LogisticRegression 메타모델
- ✅ **클래스 불균형**: scale_pos_weight=1.72
- ✅ **다중 임계값**: 5개 제출 파일 생성
- ✅ **OOF 기반 검증**

## STEP 1: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

print("✅ STEP 1: 환경 설정 완료!")

## STEP 2: 데이터 로드

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"✅ STEP 2: 데이터 로드 완료!")
print(f"   Train: {train.shape}")
print(f"   Test: {test.shape}")
print(f"   컬럼: {train.columns.tolist()}")

## STEP 3: 한글 컬럼 매핑 + 피처 엔지니어링

In [ ]:
def process_data(df, is_train=True):
    """
    한글 컬럼 매핑 + 피처 엔지니어링 (5개만)
    """
    df = df.copy()
    
    # 컬럼 매핑 딕셔너리 (한글 키워드 → 영어)
    col_map = {}
    cat_cols = []  # 범주형 컬럼
    
    for col in df.columns:
        c = col.lower()
        if 'id' in c:
            col_map[col] = 'id'
        elif '나이' in col or 'age' in c:
            col_map[col] = 'age'
        elif '키' in col and 'cm' in col:
            col_map[col] = 'height'
        elif '몸무게' in col or '체중' in col:
            col_map[col] = 'weight'
        elif 'bmi' in c:
            col_map[col] = 'bmi'
        elif '시력' in col:
            col_map[col] = 'eyesight'
        elif '충치' in col:
            col_map[col] = 'cavity'
            cat_cols.append('cavity')
        elif '혈당' in col or '공복' in col:
            col_map[col] = 'fasting_blood_sugar'
        elif '혈압' in col:
            col_map[col] = 'blood_pressure'
        elif '중성' in col:
            col_map[col] = 'triglyceride'
        elif '크레' in col:
            col_map[col] = 'serum_creatinine'
        elif '콜레스테롤' in col:
            col_map[col] = 'cholesterol'
        elif '고밀도' in col:
            col_map[col] = 'hdl'
        elif '저밀도' in col:
            col_map[col] = 'ldl'
        elif '헤모글로빈' in col:
            col_map[col] = 'hemoglobin'
        elif '단백' in col and '지단백' not in col:
            col_map[col] = 'urine_protein'
            cat_cols.append('urine_protein')
        elif '간' in col or '효소' in col:
            col_map[col] = 'gtp'
        elif 'label' in c:
            col_map[col] = 'label'
        else:
            col_map[col] = col
    
    df = df.rename(columns=col_map)
    
    # ID 제거
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    # ========== 피처 엔지니어링 (5개만) ==========
    cols = df.columns.tolist()
    
    # 1. TG/HDL 비율
    if 'triglyceride' in cols and 'hdl' in cols:
        df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
    
    # 2. 헤모글로빈 높음 (범주형)
    if 'hemoglobin' in cols:
        df['hemo_high'] = (df['hemoglobin'] > 15).astype(int)
        cat_cols.append('hemo_high')
    
    # 3. GTP 낮음 (범주형) - 낮으면 비흡연 가능성
    if 'gtp' in cols:
        df['gtp_low'] = (df['gtp'] < 1.1).astype(int)
        cat_cols.append('gtp_low')
    
    # 4. BMI 그룹 (범주형)
    if 'bmi' in cols:
        df['bmi_group'] = pd.cut(df['bmi'], bins=[0, 18.5, 23, 25, 100], labels=[0, 1, 2, 3]).astype(int)
        cat_cols.append('bmi_group')
    
    # 5. 나이 × 헤모글로빈
    if 'age' in cols and 'hemoglobin' in cols:
        df['age_x_hemo'] = df['age'] * df['hemoglobin']
    
    # 결측치 처리
    df = df.fillna(0)
    
    return df, list(set(cat_cols))

# 데이터 처리
print("🔄 STEP 3: 피처 엔지니어링 중...")
print(f"   원본 Train: {train.shape}")

train_processed, cat_cols = process_data(train, is_train=True)
test_processed, _ = process_data(test, is_train=False)

print(f"   처리 후 Train: {train_processed.shape}")
print(f"   처리 후 Test: {test_processed.shape}")
print(f"   범주형 컬럼: {cat_cols}")
print(f"\n✅ STEP 3: 피처 엔지니어링 완료!")
print(f"   피처 수: {train.shape[1]-2} → {train_processed.shape[1]-1} (+5개 생성)")

In [ ]:
# X, y 분리
X = train_processed.drop('label', axis=1)
y = train_processed['label']
X_test = test_processed.drop('label', axis=1, errors='ignore')

# 컬럼 순서 맞추기
X_test = X_test[X.columns]

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f"컬럼: {X.columns.tolist()}")

# 클래스 비율
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
scale_pos = n_neg / n_pos
print(f"\n클래스 분포: 비흡연={n_neg} ({n_neg/len(y)*100:.1f}%), 흡연={n_pos} ({n_pos/len(y)*100:.1f}%)")
print(f"scale_pos_weight: {scale_pos:.2f}")

## STEP 4: 하이퍼파라미터 튜닝

In [ ]:
print("=" * 60)
print("🔧 STEP 4: 하이퍼파라미터 튜닝")
print("=" * 60)

# CatBoost용 범주형 컬럼 (존재하는 것만)
cat_features = [c for c in cat_cols if c in X.columns]
print(f"\nCatBoost 범주형 컬럼: {cat_features}")

In [ ]:
# [1/4] XGBoost 튜닝
print("\n[1/4] XGBoost 튜닝 중...")

xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [1, 3, 5],
    'scale_pos_weight': [1.55, scale_pos, scale_pos * 1.1]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='error'),
    xgb_params, n_iter=60, cv=3, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X.values, y)
best_xgb = xgb_search.best_params_

print(f"\n✅ XGBoost 튜닝 완료")
print(f"   최적 점수: {xgb_search.best_score_:.5f}")
print(f"   최적 파라미터: {best_xgb}")

In [ ]:
# [2/4] LightGBM 튜닝
print("\n[2/4] LightGBM 튜닝 중...")

lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'scale_pos_weight': [1.55, scale_pos, scale_pos * 1.1]
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=60, cv=3, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X.values, y)
best_lgb = lgb_search.best_params_

print(f"\n✅ LightGBM 튜닝 완료")
print(f"   최적 점수: {lgb_search.best_score_:.5f}")
print(f"   최적 파라미터: {best_lgb}")

In [ ]:
# [3/4] CatBoost 튜닝
print("\n[3/4] CatBoost 튜닝 중...")

cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5],
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0, cat_features=cat_features, auto_class_weights='Balanced'),
    cat_params, n_iter=40, cv=3, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X, y)  # DataFrame 전달 (범주형 처리)
best_cat = cat_search.best_params_

print(f"\n✅ CatBoost 튜닝 완료")
print(f"   최적 점수: {cat_search.best_score_:.5f}")
print(f"   최적 파라미터: {best_cat}")

In [ ]:
# [4/4] ExtraTrees 튜닝 (⭐ 배깅 모델 추가!)
print("\n[4/4] ExtraTrees 튜닝 중...")

et_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

et_search = RandomizedSearchCV(
    ExtraTreesClassifier(random_state=42, n_jobs=-1),
    et_params, n_iter=40, cv=3, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
et_search.fit(X.values, y)
best_et = et_search.best_params_

print(f"\n✅ ExtraTrees 튜닝 완료")
print(f"   최적 점수: {et_search.best_score_:.5f}")
print(f"   최적 파라미터: {best_et}")

In [ ]:
print("\n" + "=" * 60)
print("📊 튜닝 결과 요약")
print("=" * 60)
print(f"XGBoost:    {xgb_search.best_score_:.5f}")
print(f"LightGBM:   {lgb_search.best_score_:.5f}")
print(f"CatBoost:   {cat_search.best_score_:.5f}")
print(f"ExtraTrees: {et_search.best_score_:.5f}")

## STEP 5: 5-Fold OOF 예측

In [ ]:
print("=" * 60)
print("🎯 STEP 5: 5-Fold OOF 예측")
print("=" * 60)

N_SPLITS = 5
SEED = 42

# OOF 저장
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_et = np.zeros(len(X))

# Test 저장
test_xgb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))

kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(kfold.split(X, y)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
    # XGBoost
    xgb_m = XGBClassifier(**best_xgb, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='error')
    xgb_m.fit(X_tr.values, y_tr)
    oof_xgb[va_idx] = xgb_m.predict_proba(X_va.values)[:, 1]
    test_xgb += xgb_m.predict_proba(X_test.values)[:, 1] / N_SPLITS
    
    # LightGBM
    lgb_m = LGBMClassifier(**best_lgb, random_state=SEED, verbose=-1)
    lgb_m.fit(X_tr.values, y_tr)
    oof_lgb[va_idx] = lgb_m.predict_proba(X_va.values)[:, 1]
    test_lgb += lgb_m.predict_proba(X_test.values)[:, 1] / N_SPLITS
    
    # CatBoost (DataFrame)
    cat_m = CatBoostClassifier(**best_cat, random_state=SEED, verbose=0, cat_features=cat_features, auto_class_weights='Balanced')
    cat_m.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat_m.predict_proba(X_va)[:, 1]
    test_cat += cat_m.predict_proba(X_test)[:, 1] / N_SPLITS
    
    # ExtraTrees
    et_m = ExtraTreesClassifier(**best_et, random_state=SEED, n_jobs=-1)
    et_m.fit(X_tr.values, y_tr)
    oof_et[va_idx] = et_m.predict_proba(X_va.values)[:, 1]
    test_et += et_m.predict_proba(X_test.values)[:, 1] / N_SPLITS
    
    # Fold 성능
    print(f"   XGB: {accuracy_score(y_va, (oof_xgb[va_idx]>=0.5).astype(int)):.4f}")
    print(f"   LGB: {accuracy_score(y_va, (oof_lgb[va_idx]>=0.5).astype(int)):.4f}")
    print(f"   CAT: {accuracy_score(y_va, (oof_cat[va_idx]>=0.5).astype(int)):.4f}")
    print(f"   ET:  {accuracy_score(y_va, (oof_et[va_idx]>=0.5).astype(int)):.4f}")

print("\n✅ STEP 5: OOF 예측 완료!")

In [ ]:
# 개별 모델 OOF 성능
print("\n" + "=" * 60)
print("📊 개별 모델 OOF 성능 (threshold=0.5)")
print("=" * 60)

models_oof = {
    'XGBoost': oof_xgb,
    'LightGBM': oof_lgb,
    'CatBoost': oof_cat,
    'ExtraTrees': oof_et
}

for name, oof in models_oof.items():
    pred = (oof >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    print(f"{name:12} | Accuracy: {acc:.5f} | F1: {f1:.5f}")

## STEP 6: Stacking 앙상블

In [ ]:
print("=" * 60)
print("🔧 STEP 6: Stacking 앙상블")
print("=" * 60)

# OOF 예측을 새 피처로
oof_features = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_et])
test_features = np.column_stack([test_xgb, test_lgb, test_cat, test_et])

print(f"OOF features: {oof_features.shape}")
print(f"Test features: {test_features.shape}")

# 메타 모델 (LogisticRegression)
meta_model = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
meta_model.fit(oof_features, y)

# Stacking 예측
oof_meta = meta_model.predict_proba(oof_features)[:, 1]
test_meta = meta_model.predict_proba(test_features)[:, 1]

# 메타모델 가중치
print(f"\n📊 메타모델 가중치:")
weights = meta_model.coef_[0]
for name, w in zip(['XGBoost', 'LightGBM', 'CatBoost', 'ExtraTrees'], weights):
    print(f"   {name}: {w:+.4f}")

# Stacking OOF 성능
stack_pred = (oof_meta >= 0.5).astype(int)
stack_acc = accuracy_score(y, stack_pred)
stack_f1 = f1_score(y, stack_pred)

print(f"\n🏆 Stacking OOF 성능:")
print(f"   Accuracy: {stack_acc:.5f}")
print(f"   F1-Score: {stack_f1:.5f}")

## STEP 7: 최적 임계값 탐색

In [ ]:
print("=" * 60)
print("🔍 STEP 7: 최적 임계값 탐색")
print("=" * 60)

best_th = 0.5
best_acc = 0
results = []

for th in np.arange(0.40, 0.60, 0.01):
    pred = (oof_meta >= th).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': th, 'accuracy': acc, 'f1': f1})
    if acc > best_acc:
        best_acc = acc
        best_th = th

results_df = pd.DataFrame(results)
print("\n상위 10개 임계값:")
print(results_df.nlargest(10, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_th:.2f}")
print(f"   OOF Accuracy: {best_acc:.5f}")

## STEP 8: 다중 임계값 제출 파일 생성

In [ ]:
print("=" * 60)
print("📝 STEP 8: 다중 임계값 제출 파일 생성")
print("=" * 60)

# 5개 임계값
thresholds = [0.45, 0.47, 0.50, 0.52, 0.55]

# 최적 임계값도 포함
if best_th not in thresholds:
    thresholds.append(round(best_th, 2))
thresholds = sorted(set(thresholds))

file_paths = []

for th in thresholds:
    # 예측
    pred = (test_meta >= th).astype(int)
    
    # OOF 성능
    oof_pred = (oof_meta >= th).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    # 제출 파일
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    # 저장
    th_str = str(int(th * 100))
    filename = f'submission_v8_t{th_str}.csv'
    filepath = result_path + filename
    sub.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    # 분포
    n_smoking = (pred == 1).sum()
    pct_smoking = n_smoking / len(pred) * 100
    
    marker = "⭐" if th == best_th else "  "
    print(f"\n{marker} {filename}")
    print(f"   임계값: {th:.2f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   예측 분포: 비흡연={len(pred)-n_smoking} ({100-pct_smoking:.1f}%), 흡연={n_smoking} ({pct_smoking:.1f}%)")

print(f"\n✅ {len(thresholds)}개 제출 파일 생성 완료!")

In [ ]:
# 검증
print("\n🔍 제출 파일 검증:")
for fp in file_paths:
    df = pd.read_csv(fp)
    filename = fp.split('/')[-1]
    valid = df['label'].dtype in ['int64', 'int32'] and set(df['label'].unique()).issubset({0, 1})
    status = "✅" if valid else "❌"
    print(f"   {status} {filename}: shape={df.shape}, dtype={df['label'].dtype}")

## STEP 9: 다운로드

In [ ]:
from google.colab import files

# 최적 임계값 파일 다운로드
best_file = result_path + f'submission_v8_t{int(best_th*100)}.csv'
files.download(best_file)

print("\n" + "=" * 60)
print("🎉 V8 Stacking 앙상블 완료!")
print("=" * 60)
print(f"\n📊 최종 결과:")
print(f"   모델: XGBoost + LightGBM + CatBoost + ExtraTrees")
print(f"   메타모델: LogisticRegression")
print(f"   최적 임계값: {best_th:.2f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"\n📁 생성된 파일:")
for fp in file_paths:
    marker = "👉" if f't{int(best_th*100)}' in fp else "  "
    print(f"   {marker} {fp.split('/')[-1]}")
print(f"\n🚀 먼저 ⭐ 최적 임계값 파일을 제출하세요!")

In [ ]:
# 다른 파일도 다운로드 (선택)
print("\n📥 다른 임계값 파일 다운로드:")
for fp in file_paths:
    if fp != best_file:
        files.download(fp)
        print(f"   ✅ {fp.split('/')[-1]}")